In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

In [3]:

pio.renderers.default = "browser"

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData\xyz_Smooth"

BASE_CSV_DIR  = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Chase_CSVData"
BASE_PLOT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Chase_Data"

GLOBAL_CSV_DIR = os.path.join(BASE_CSV_DIR, "global")
LOCAL_CSV_DIR  = os.path.join(BASE_CSV_DIR, "local")
os.makedirs(GLOBAL_CSV_DIR, exist_ok=True)
os.makedirs(LOCAL_CSV_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv")))
if not csv_files:
    raise FileNotFoundError("No *_SMOOTH.csv files found")

# ==========================================================
# POINT DEFINITIONS
# ==========================================================
POINT_IDS = ['pt2','pt3','pt4','pt5','pt6','pt7']
LABELS    = ['Head','L_WB','R_WB','L_WT','R_WT','Abdomen']
SKELETON  = [(0,5),(0,1),(0,2),(1,2),(1,3),(2,4),(5,1),(5,2)]
AXES = ['X','Y','Z']

# ==========================================================
# UTILITIES
# ==========================================================
def safe_norm(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = np.nan
    return v / n

def build_body_axes(center, head, left, right):
    y = safe_norm(head - center)          # forward
    x = safe_norm(right - left)           # left-right
    z = safe_norm(np.cross(x, y))         # RIGHT-HANDED
    return x, y, z



def transform_global_to_local(points, R, center):
    p_rel = points - center[:, None, :]
    return np.einsum("nij,nkj->nki", R, p_rel)

def add_animation_controls(fig, frames):
    fig.update_layout(
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            x=0.05,
            y=1.15,
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[None, {
                        "frame": {"duration": 30, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 0}
                    }]
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[[None], {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }]
                )
            ]
        )],
        sliders=[dict(
            active=0,
            x=0.1,
            y=0.05,
            len=0.85,
            currentvalue=dict(prefix="Frame: "),
            steps=[
                dict(
                    method="animate",
                    args=[[str(frames[i])], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": True},
                        "transition": {"duration": 0}
                    }],
                    label=str(frames[i])
                )
                for i in range(len(frames))
            ]
        )]
    )

# ==========================================================
# PROCESS TRIALS
# ==========================================================
for path in csv_files:

    base = os.path.splitext(os.path.basename(path))[0]
    print("\nProcessing:", base)

    # ------------------------------------------------------
    # Extract Trial + RPM
    # ------------------------------------------------------
    m = re.search(r"(Trial\d+_\d+rpm)", base)
    if not m:
        raise ValueError(f"Could not extract Trial+RPM from filename: {base}")
    trial_name = m.group(1)

    # Trial-specific plot folder
    plot_out_dir = os.path.join(BASE_PLOT_DIR, trial_name)
    os.makedirs(plot_out_dir, exist_ok=True)

    # ------------------------------------------------------
    # Load data
    # ------------------------------------------------------
    df = pd.read_csv(path)
    frames = df["frame"].values if "frame" in df.columns else np.arange(len(df))

    def get_marker(pid):
        return df[[f"{pid}_X", f"{pid}_Y", f"{pid}_Z"]].values

    head = get_marker("pt2")
    lwb  = get_marker("pt3")
    rwb  = get_marker("pt4")
    center = (lwb + rwb) / 2

    global_pts = df[[f"{pid}_{ax}" for pid in POINT_IDS for ax in AXES]].values
    global_pts = global_pts.reshape(len(df), len(POINT_IDS), 3)

    x_axis, y_axis, z_axis = build_body_axes(center, head, lwb, rwb)

    R = np.zeros((len(df), 3, 3))
    R[:,0,:] = x_axis
    R[:,1,:] = y_axis
    R[:,2,:] = z_axis

    local_pts = transform_global_to_local(global_pts, R, center)

    # ------------------------------------------------------
    # SAVE CSVs (GLOBAL + LOCAL)
    # ------------------------------------------------------
    coord_cols = [f"{pid}_{ax}" for pid in POINT_IDS for ax in AXES]

    global_df = pd.DataFrame(
        global_pts.reshape(len(df), -1),
        columns=[f"{c}_global" for c in coord_cols]
    )
    global_df.insert(0, "frame", frames)
    global_df.to_csv(
        os.path.join(GLOBAL_CSV_DIR, f"{trial_name}_global_coordinates.csv"),
        index=False
    )

    local_df = pd.DataFrame(
        local_pts.reshape(len(df), -1),
        columns=[f"{c}_local" for c in coord_cols]
    )
    local_df.insert(0, "frame", frames)
    local_df.to_csv(
        os.path.join(LOCAL_CSV_DIR, f"{trial_name}_local_coordinates.csv"),
        index=False
    )

    # ------------------------------------------------------
    # GLOBAL VIEW
    # ------------------------------------------------------
    global_frames = []
    for i in range(len(df)):
        traces = []
        for a,b in SKELETON:
            traces.append(go.Scatter3d(
                x=[global_pts[i,a,0], global_pts[i,b,0]],
                y=[global_pts[i,a,1], global_pts[i,b,1]],
                z=[global_pts[i,a,2], global_pts[i,b,2]],
                mode="lines",
                line=dict(color="gray", width=4),
                showlegend=False
            ))
        traces.append(go.Scatter3d(
            x=global_pts[i,:,0],
            y=global_pts[i,:,1],
            z=global_pts[i,:,2],
            mode="markers+text",
            text=LABELS,
            marker=dict(size=6, color="cyan"),
            showlegend=False
        ))
        global_frames.append(go.Frame(data=traces, name=str(frames[i])))

    fig_global = go.Figure(data=global_frames[0].data, frames=global_frames)
    fig_global.update_layout(
        title=f"{trial_name} — GLOBAL VIEW",
        scene=dict(aspectmode="data")
    )
    add_animation_controls(fig_global, frames)
    fig_global.write_html(os.path.join(plot_out_dir, "global_view.html"))

    # ------------------------------------------------------
    # LOCAL VIEW
    # ------------------------------------------------------
    local_frames = []
    for i in range(len(df)):
        traces = []
        for a,b in SKELETON:
            traces.append(go.Scatter3d(
                x=[local_pts[i,a,0], local_pts[i,b,0]],
                y=[local_pts[i,a,1], local_pts[i,b,1]],
                z=[local_pts[i,a,2], local_pts[i,b,2]],
                mode="lines",
                line=dict(color="gray", width=4),
                showlegend=False
            ))
        traces.append(go.Scatter3d(
            x=local_pts[i,:,0],
            y=local_pts[i,:,1],
            z=local_pts[i,:,2],
            mode="markers+text",
            text=LABELS,
            marker=dict(size=6, color="orange"),
            # showlegend=False
            
        ))
        local_frames.append(go.Frame(data=traces, name=str(frames[i])))

    fig_local = go.Figure(data=local_frames[0].data, frames=local_frames)
    fig_local.update_layout(
        title=f"{trial_name} — LOCAL (BODY-CENTRIC) VIEW",
        scene=dict(aspectmode="data")
    )
    add_animation_controls(fig_local, frames)
    fig_local.write_html(os.path.join(plot_out_dir, "local_view.html"))

print("\n ALL TRIALS PROCESSED SUCCESSFULLY")





Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_180rpmxyzpts_SMOOTH

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_SMOOTH

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_SMOOTH

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_SMOOTH

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_SMOOTH

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_400rpmxyzpts_SMOOTH

Processing: Final_DLTdv87_Data_DLTdv8_data_Trial7_400rpmxyzpts_SMOOTH

 ALL TRIALS PROCESSED SUCCESSFULLY


# Sanity check

In [17]:
import numpy as np
import pandas as pd


# ---------------------------
# LOAD MOCK CSV
# ---------------------------
mock_csv = r"C:\Users\munpa\Downloads\mock_global2local.csv"
df = pd.read_csv(mock_csv)

# ---------------------------
# EXTRACT MARKERS
# ---------------------------
head = df[["pt2_X","pt2_Y","pt2_Z"]].values
lwb  = df[["pt3_X","pt3_Y","pt3_Z"]].values
rwb  = df[["pt4_X","pt4_Y","pt4_Z"]].values

center = (lwb + rwb) / 2   # should be (0,0,0)

# ---------------------------
# BUILD BODY AXES (REAL CODE)
# ---------------------------
x_axis, y_axis, z_axis = build_body_axes(center, head, lwb, rwb)

R = np.zeros((len(df), 3, 3))
R[:,0,:] = x_axis
R[:,1,:] = y_axis
R[:,2,:] = z_axis

# ---------------------------
# STACK POINTS
# ---------------------------
POINT_IDS = ["pt2","pt3","pt4","pt5"]

points = np.stack(
    [df[[f"{p}_X", f"{p}_Y", f"{p}_Z"]].values for p in POINT_IDS],
    axis=1
)

# ---------------------------
# GLOBAL → LOCAL
# ---------------------------
local_pts = transform_global_to_local(points, R, center)

# ---------------------------
# PRINT RESULTS
# ---------------------------
print("\nLocal coordinates (rounded):")
for lab, p in zip(POINT_IDS, local_pts[0]):
    print(f"{lab:4s} → {np.round(p, 3)}")
    
    
    
def angle_deg(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.degrees(np.arccos(np.clip(np.dot(a, b), -1, 1)))

global_axes = {
    "Xg": np.array([1,0,0]),
    "Yg": np.array([0,1,0]),
    "Zg": np.array([0,0,1]),
}

local_axes = {
    "Xl": x_axis[0],
    "Yl": y_axis[0],
    "Zl": z_axis[0],
}

print("\nAngles (degrees):")
for g_name, g_vec in global_axes.items():
    for l_name, l_vec in local_axes.items():
        ang = angle_deg(g_vec, l_vec)
        print(f"{g_name} → {l_name}: {ang:6.2f}")




Local coordinates (rounded):
pt2  → [0. 1. 0.]
pt3  → [-1.  0.  0.]
pt4  → [1. 0. 0.]
pt5  → [ 0. -1.  0.]

Angles (degrees):
Xg → Xl:   0.00
Xg → Yl:  90.00
Xg → Zl:  90.00
Yg → Xl:  90.00
Yg → Yl:   0.00
Yg → Zl:  90.00
Zg → Xl:  90.00
Zg → Yl:  90.00
Zg → Zl:   0.00
